In [8]:
import pandas as pd

# Load CSV
df = pd.read_csv("ams_features.csv")

# Keep only the columns you need
cols_to_keep = [
    "buurtnaam",
    "temp_mean",
    "buurt_area",
    "ndvi_mean",
    "water_prc",
    "road_prc",
]

df = df[cols_to_keep].copy()

# 3. Check the result
print(df.head())
print(df.info())

# 4. Save cleaned version
df.to_csv("heat_features_clean.csv", index=False)

             buurtnaam  temp_mean  buurt_area  ndvi_mean  water_prc  road_prc
0  Steigereiland Noord  28.528717  220003.698   0.239546     34.909     0.035
1               Borneo  29.501901  267367.961   0.088677     16.420     0.083
2     Westerdokseiland  31.378875  337889.098   0.037265     15.461     0.001
3    Willibrordusbuurt  37.325497  123949.349   0.223259     13.415     0.061
4    Nieuwendijk Noord  38.112997   66013.804   0.074301     11.489     0.004
<class 'pandas.DataFrame'>
RangeIndex: 481 entries, 0 to 480
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   buurtnaam   479 non-null    str    
 1   temp_mean   481 non-null    float64
 2   buurt_area  481 non-null    float64
 3   ndvi_mean   481 non-null    float64
 4   water_prc   481 non-null    float64
 5   road_prc    479 non-null    float64
dtypes: float64(5), str(1)
memory usage: 22.7 KB
None


In [9]:
pip install statsmodels


Note: you may need to restart the kernel to use updated packages.


In [10]:

import statsmodels.api as sm

# Load data
df = pd.read_csv("heat_features_clean.csv")

# Remove missing values
df = df.dropna()

# Remove rows with negative values
df = df[
    (df["temp_mean"] >= 0) &
    (df["ndvi_mean"] >= 0) &
    (df["water_prc"] >= 0) &
    (df["road_prc"] >= 0)
]

# Define target
y = df["temp_mean"]

# Define predictors
X = df[[
    "ndvi_mean",
    "water_prc",
    "road_prc"
]]

# Add intercept
X = sm.add_constant(X)

# Fit model
model = sm.OLS(y, X).fit()

# Results
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:              temp_mean   R-squared:                       0.154
Model:                            OLS   Adj. R-squared:                  0.148
Method:                 Least Squares   F-statistic:                     28.40
Date:                Mon, 01 Jun 2026   Prob (F-statistic):           6.90e-17
Time:                        11:49:50   Log-Likelihood:                -1079.1
No. Observations:                 473   AIC:                             2166.
Df Residuals:                     469   BIC:                             2183.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         38.4683      0.345    111.530      0.0

In [ ]:
import geopandas as gpd

gdf = gpd.read_file("heat_ams.gpkg")


gdf["x"] = gdf.geometry.centroid.x
gdf["y"] = gdf.geometry.centroid.y


Empty GeoDataFrame
Columns: [buurtcode, buurtnaam, wijkcode, gemeentecode, gemeentenaam, indelingswijzigingWijkenEnBuurten, water, meestVoorkomendePostcode, dekkingspercentage, omgevingsadressendichtheid, stedelijkheidAdressenPerKm2, bevolkingsdichtheidInwonersPerKm2, aantalInwoners, mannen, vrouwen, percentagePersonen0Tot15Jaar, percentagePersonen15Tot25Jaar, percentagePersonen25Tot45Jaar, percentagePersonen45Tot65Jaar, percentagePersonen65JaarEnOuder, percentageOngehuwd, percentageGehuwd, percentageGescheid, percentageVerweduwd, geboorteTotaal, geboortesPer1000Inwoners, sterfteTotaal, sterfteRelatief, aantalHuishoudens, percentageEenpersoonshuishoudens, percentageHuishoudensZonderKinderen, percentageHuishoudensMetKinderen, gemiddeldeHuishoudsgrootte, percentageWesterseMigratieachtergrond, percentageNietWesterseMigratieachtergrond, percentageUitMarokko, percentageUitNederlandseAntillenEnAruba, percentageUitSuriname, percentageUitTurkije, percentageOverigeNietwestersemigratieachtergron

In [18]:
print(fiona.listlayers("heat_ams.gpkg"))


['a']


   ---------------------------------------- 0.0/24.5 MB ? eta -:--:--
   --- ------------------------------------ 2.4/24.5 MB 11.2 MB/s eta 0:00:02
   --------- ------------------------------ 5.5/24.5 MB 13.4 MB/s eta 0:00:02
   ------------ --------------------------- 7.3/24.5 MB 12.2 MB/s eta 0:00:02
   ---------------- ----------------------- 10.0/24.5 MB 12.4 MB/s eta 0:00:02
   --------------------- ------------------ 13.1/24.5 MB 12.4 MB/s eta 0:00:01
   --------------------------- ------------ 16.8/24.5 MB 13.4 MB/s eta 0:00:01
   --------------------------------- ------ 20.2/24.5 MB 13.6 MB/s eta 0:00:01
   ------------------------------------- -- 23.1/24.5 MB 13.6 MB/s eta 0:00:01
   ---------------------------------------- 24.5/24.5 MB 13.0 MB/s  0:00:01

   ---------------------------------------- 0/5 [click]
   -------- ------------------------------- 1/5 [attrs]
   -------- ------------------------------- 1/5 [attrs]
   -------------------------------- ------- 4/5 [fiona]


In [12]:
import numpy as np

coords = list(zip(df["x"], df["y"]))

y = df["temp_mean"].values.reshape((-1,1))

X = df[[
    "ndvi_mean",
    "water_prc",
    "road_prc"
]].values

KeyError: 'x'

In [ ]:
from mgwr.sel_bw import Sel_BW

bw = Sel_BW(coords, y, X).search()

print(bw)

In [ ]:
from mgwr.gwr import GWR

gwr = GWR(coords, y, X, bw)

results = gwr.fit()

print(results.summary())

In [ ]:
gdf["ndvi_coef"] = results.params[:,1]

gdf.plot(
    column="ndvi_coef",
    legend=True
)